# MODULE B - Data Engineering
#### Objective: The objective of Module B is to design, implement, and validate a reproducible, scalable, and auditable data engineering pipeline that transforms raw IBM AMLSim transaction data into clean, structured, and analysis-ready datasets for coordinated fraud ring detection and investigation.The pipeline must reliably process approximately 100,000 accounts, 12.48 million transactions, and 17,052 alert records, while preserving transaction integrity, account relationships, temporal ordering, and the original synthetic fraud labels.The module will focus on five engineering components:
#### B1. Raw data ingestion 
#### B2. Schema validation 
#### B3. Data cleaning and quarantine 
#### B4. SQL analytical storage 
#### B5. Reproducible pipeline


# B1 -Raw Data Ingestion
#### Objective: Develop a memory-efficient ingestion process for reading large CSV files, preserving source data, standardizing input handling, and recording ingestion metadata.

In [37]:
#B1.1 Recursive raw dataset inventory
from pathlib import Path
from datetime import datetime, timezone
import csv
import stat
DATASET_ROOT = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges")
OUTPUT_PATH = Path(r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv")

FIELDNAMES = [
    "relative_path",
    "file_name",
    "extension",
    "size_bytes",
    "last_modified_utc",
]
def validate_paths(root: Path, output: Path) -> None:
    root=root.resolve()
    output=output.resolve()
    if not root.exists():
        raise FileNotFoundError(f'Dataset root not exist: {root}')
    if not root.is_dir():
        raise NotADirectoryError(f'Dataset root is not a directory : {root}')
    if output.suffix.lower() != ".csv" :
        raise ValueError("Output file must have a .csv extension")
    try:
       output.relative_to(root)
    except ValueError:
        pass
    else:
        raise ValueError("Output file cannot be inside the dataset root")
import os
from pathlib import Path

def discover_files(root: Path) -> list[Path]:
    files = []

    def handle_error(error):
        raise OSError(
            f"Failed to enumerate directory: {error.filename}"
        ) from error

    for dirpath, dirnames, filenames in os.walk(
        root,
        topdown=True,
        onerror=handle_error,
        followlinks=False
    ):
        current_dir = Path(dirpath)

        for filename in filenames:
            file_path = current_dir / filename

            try:
                file_stat = file_path.lstat()
            except OSError as e:
                raise OSError(f"Failed to retrieve metadata for {file_path}") from e

            if stat.S_ISLNK(file_stat.st_mode):
                continue

            if stat.S_ISREG(file_stat.st_mode):
                files.append(file_path)

    files.sort(
        key=lambda file: file.relative_to(root).as_posix()
    )

    return files
    

    
def extract_metadata(file_path: Path, root: Path) -> dict:
    #get file metadata once
    try:
        file_stats=file_path.stat()
    except OSError as e:
        raise OSError(f"failed to retrieve metadata for {file_path}") from e
    #POSIX - style relative path
    relative_path=file_path.relative_to(root).as_posix()
    #file name and extension
    file_name=file_path.name
    extension=file_path.suffix
    #file size in bytes
    size_bytes=file_stats.st_size
    # UTC ISO 8601 timestamp
    last_modified_utc = datetime.fromtimestamp(
        file_stats.st_mtime,
        tz=timezone.utc
    ).isoformat(timespec="microseconds")

    # Return metadata dictionary
    return {
        "relative_path": relative_path,
        "file_name": file_name,
        "extension": extension,
        "size_bytes": size_bytes,
        "last_modified_utc": last_modified_utc
    }
def build_manifest(root: Path) -> list[dict]:
    rows = []

    # Iterate through all discovered files
    for file_path in discover_files(root):
        rows.append(extract_metadata(file_path, root))

    return rows
def write_manifest(rows: list[dict], output: Path) -> None:
    # create parent directory if needed
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w",encoding="utf-8",newline="") as file:
        #create CSV writer
        writer=csv.DictWriter(file,fieldnames=FIELDNAMES,lineterminator="\n")
        #write header
        writer.writeheader()
        #write all rows
        writer.writerows(rows)


    

In [38]:
#main execution cell of B1.1
validate_paths(DATASET_ROOT, OUTPUT_PATH)

rows = build_manifest(DATASET_ROOT)

write_manifest(rows, OUTPUT_PATH)

print(f"Manifest created: {OUTPUT_PATH}")
print(f"Files inventoried: {len(rows)}")

Manifest created: C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\reports\dataset_manifest.csv
Files inventoried: 14


In [39]:
#repeatibility testing of B1.1
manifest1 = build_manifest(DATASET_ROOT)
manifest2 = build_manifest(DATASET_ROOT)

assert manifest1 == manifest2

print("Repeatability test passed")


Repeatability test passed


#### Failure Scenario Testing of section B1.1

In [40]:
#test 1 : missing dataset
try:
    validate_paths(
        Path("missing_folder"),
        OUTPUT_PATH
    )
except FileNotFoundError:
    print("PASS")

PASS


In [41]:
#test 2 : Output not csv
try:
    validate_paths(
        DATASET_ROOT,
        Path("report.txt")
    )
except ValueError:
    print("PASS")

PASS


In [42]:
#test 3 : Ouput inside dataset
try:
    validate_paths(
        DATASET_ROOT,
        DATASET_ROOT / "manifest.csv"
    )
except ValueError:
    print("PASS")

PASS


In [43]:
#test 4 : metadata failure
try:
    extract_metadata(
        Path("does_not_exist.csv"),
        DATASET_ROOT
    )
except OSError:
    print("PASS")

PASS


In [44]:
#test 4 : directory-scan failure
import os

original_walk = os.walk

def failing_walk(*args, **kwargs):
    kwargs["onerror"](
        OSError("Simulated directory scan failure")
    )
    return []

os.walk = failing_walk

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Directory-scan failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    os.walk = original_walk

PASS: Directory-scan failure detected


In [45]:
#test - 5:discovery stage metadata failure
# TEST: Discovery-stage metadata failure

from pathlib import Path

original_lstat = Path.lstat

def failing_lstat(self):
    raise OSError(
        "Simulated metadata access failure"
    )

Path.lstat = failing_lstat

try:
    discover_files(DATASET_ROOT)
except OSError:
    print("PASS: Metadata failure detected")
else:
    raise AssertionError(
        "FAIL: Expected OSError was not raised"
    )
finally:
    Path.lstat = original_lstat



PASS: Metadata failure detected


In [9]:
# B1.2 - Header, Schema and Sample Inspection
# B1.2.1 - transactions.csv inspection

import csv
from pathlib import Path


def inspect_transaction_csv(csv_path):
    try:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            reader = csv.reader(file)

            # Handle empty file
            try:
                header = next(reader)
            except StopIteration:
                print("CSV file is empty")
                return

            print("Header:")
            print(header)

            print("\nNumber of Columns:")
            print(len(header))

            print("\nFirst 3 Records:")

            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )

    except FileNotFoundError:
        print(f"Error: File not found -> {csv_path}")

    except PermissionError:
        print(f"Error: Permission denied -> {csv_path}")

In [11]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
inspect_transaction_csv(csv_path)

Header:
['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID']

Number of Columns:
8

First 3 Records:

Record 1:
['1', '5942', '92982', 'TRANSFER', '516.95', '0', 'False', '-1']
Field Count:
8

Record 2:
['2', '86700', '43995', 'TRANSFER', '197.88', '0', 'False', '-1']
Field Count:
8

Record 3:
['3', '86700', '95516', 'TRANSFER', '197.88', '0', 'False', '-1']
Field Count:
8


In [14]:
# B1.2 - Header, Schema and Sample Inspection
# B1.2.1 accounts.csv inspection
def inspect_accounts_csv(account_csv_path):
    try:
        with open(account_csv_path,"r",encoding='utf-8',newline="") as file:
            reader=csv.reader(file)
            try:
                header=next(reader)
            except StopIteration:
                print(f"file is empty:{account_csv_path}")
            print("Header:")
            print(header)
            print("\nNumber Of Columns:")
            print(len(header))
            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )


    except FileNotFoundError:
        print(f"Error:File not found - {account_csv_path}")
    except PermissionError:
        print(f"Error:Permission denied - {account_csv_path}")       

In [15]:
account_csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv"
inspect_accounts_csv(account_csv_path)


Header:
['ACCOUNT_ID', 'CUSTOMER_ID', 'INIT_BALANCE', 'COUNTRY', 'ACCOUNT_TYPE', 'IS_FRAUD', 'TX_BEHAVIOR_ID']

Number Of Columns:
7

Record 1:
['0', 'C_0', '184.44', 'US', 'I', 'false', '1']
Field Count:
7

Record 2:
['1', 'C_1', '175.80', 'US', 'I', 'false', '1']
Field Count:
7

Record 3:
['2', 'C_2', '142.06', 'US', 'I', 'false', '1']
Field Count:
7


In [16]:
#B1.2 - Header, Schema and Sample Inspection
#B1.2.3 - alerts.csv inspection
#Generalization of the previous inspection function
import csv
from pathlib import Path


def inspect_csv(csv_path):
    try:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            reader = csv.reader(file)

            # Handle empty file
            try:
                header = next(reader)
            except StopIteration:
                print("CSV file is empty")
                return

            print("File:", csv_path)
            print("\nHeader:")
            print(header)

            print("\nNumber of Columns:")
            print(len(header))

            print("\nFirst 3 Records:")

            for i in range(3):
                try:
                    row = next(reader)
                except StopIteration:
                    break

                print(f"\nRecord {i + 1}:")
                print(row)

                print("Field Count:")
                print(len(row))

                # Report field-count mismatch
                if len(row) != len(header):
                    print(
                        f"Observation: Header has {len(header)} fields "
                        f"but this record has {len(row)} fields"
                    )

    except FileNotFoundError:
        print(f"Error: File not found -> {csv_path}")

    except PermissionError:
        print(f"Error: Permission denied -> {csv_path}")

In [18]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
inspect_csv(csv_path)

File: C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv

Header:
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']

Number of Columns:
9

First 3 Records:

Record 1:
['2036', 'cycle', 'True', '188', '24830', '33880', 'TRANSFER', '12.4', '0']
Field Count:
9

Record 2:
['842', 'fan_in', 'True', '556', '45959', '91430', 'TRANSFER', '3.07', '0']
Field Count:
9

Record 3:
['3000', 'cycle', 'True', '1038', '25885', '22925', 'TRANSFER', '13.06', '0']
Field Count:
9


In [10]:
#B1.3 - Data Types and Memory Assessment
# B1.3.1 — Transaction.csv memory baseline
import pandas as pd
def memory_baseline(csv_path):
    try:
        
        df = pd.read_csv(
            csv_path,
            dtype=str,
            nrows=100000,
            keep_default_na=False
        )

        print("SAMPLE SIZE:")
        print(f"Rows: {len(df)}")
        print(f"Columns: {len(df.columns)}")

        print("\nCOLUMN NAMES")
        print(list(df.columns))

        print("\nOBSERVED DATA TYPES")
        print(df.dtypes)

        print("\nMEMORY USAGE PER COLUMN (bytes)")
        memory_per_column = df.memory_usage(deep=True)
        print(memory_per_column)

        total_memory_bytes = memory_per_column.sum()
        total_memory_mib = total_memory_bytes / (1024 * 1024)

        print("\nTOTAL MEMORY")
        print(f"Bytes: {total_memory_bytes:,}")
        print(f"MiB: {total_memory_mib:.2f}")

        avg_memory_per_row = (
            total_memory_bytes / len(df)
            if len(df) > 0
            else 0
        )

        print("\nAVERAGE MEMORY PER ROW")
        print(f"{avg_memory_per_row:.2f} bytes")

    except FileNotFoundError:
        print(f"File not found: {csv_path}")
        
    except pd.errors.EmptyDataError:
       print("CSV file is empty")
    

In [11]:
#memory baseline for transaction.csv
transaction_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
csv_path=transaction_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 100000
Columns: 8

COLUMN NAMES
['TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP', 'IS_FRAUD', 'ALERT_ID']

OBSERVED DATA TYPES
TX_ID                  object
SENDER_ACCOUNT_ID      object
RECEIVER_ACCOUNT_ID    object
TX_TYPE                object
TX_AMOUNT              object
TIMESTAMP              object
IS_FRAUD               object
ALERT_ID               object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                      132
TX_ID                  5388895
SENDER_ACCOUNT_ID      5388249
RECEIVER_ACCOUNT_ID    5388601
TX_TYPE                5700000
TX_AMOUNT              5442638
TIMESTAMP              5000000
IS_FRAUD               5399856
ALERT_ID               5100266
dtype: int64

TOTAL MEMORY
Bytes: 42,808,637
MiB: 40.83

AVERAGE MEMORY PER ROW
428.09 bytes


In [14]:
#memory baseline for accounts.csv
account_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv"
csv_path=account_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 100000
Columns: 7

COLUMN NAMES
['ACCOUNT_ID', 'CUSTOMER_ID', 'INIT_BALANCE', 'COUNTRY', 'ACCOUNT_TYPE', 'IS_FRAUD', 'TX_BEHAVIOR_ID']

OBSERVED DATA TYPES
ACCOUNT_ID        object
CUSTOMER_ID       object
INIT_BALANCE      object
COUNTRY           object
ACCOUNT_TYPE      object
IS_FRAUD          object
TX_BEHAVIOR_ID    object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                 132
ACCOUNT_ID        5388890
CUSTOMER_ID       5588890
INIT_BALANCE      5500000
COUNTRY           5100000
ACCOUNT_TYPE      5000000
IS_FRAUD          5382909
TX_BEHAVIOR_ID    5000000
dtype: int64

TOTAL MEMORY
Bytes: 36,960,821
MiB: 35.25

AVERAGE MEMORY PER ROW
369.61 bytes


In [15]:
#memory baseline for alerts.csv
alert_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
csv_path=alert_path
memory_baseline(csv_path)

SAMPLE SIZE:
Rows: 17052
Columns: 9

COLUMN NAMES
['ALERT_ID', 'ALERT_TYPE', 'IS_FRAUD', 'TX_ID', 'SENDER_ACCOUNT_ID', 'RECEIVER_ACCOUNT_ID', 'TX_TYPE', 'TX_AMOUNT', 'TIMESTAMP']

OBSERVED DATA TYPES
ALERT_ID               object
ALERT_TYPE             object
IS_FRAUD               object
TX_ID                  object
SENDER_ACCOUNT_ID      object
RECEIVER_ACCOUNT_ID    object
TX_TYPE                object
TX_AMOUNT              object
TIMESTAMP              object
dtype: object

MEMORY USAGE PER COLUMN (bytes)
Index                     132
ALERT_ID               899626
ALERT_TYPE             928347
IS_FRAUD               903756
TX_ID                  956895
SENDER_ACCOUNT_ID      918945
RECEIVER_ACCOUNT_ID    918941
TX_TYPE                971964
TX_AMOUNT              911436
TIMESTAMP              877257
dtype: int64

TOTAL MEMORY
Bytes: 8,287,299
MiB: 7.90

AVERAGE MEMORY PER ROW
486.00 bytes


In [ ]:
#B1.3 - Data Types and Memory Assessment
# B1.3.2 — Building a memory estimation utility
def estimate_memory(sample_rows,sample_memory_bytes,full_dataset_rows,observed_memory_bytes):
    if sample_rows <= 0:
        raise ValueError("sample_rows must be greater than zero")
    if observed_memory_bytes <= 0:
        raise ValueError(
            "observed_memory_bytes must be greater than zero"
        )
    if min(sample_memory_bytes, full_dataset_rows, observed_memory_bytes) < 0:
        raise ValueError("Inputs cannot be negative")

    avg_memory_per_row = sample_memory_bytes / sample_rows
    estimated_full_memory_bytes = avg_memory_per_row * full_dataset_rows
    estimated_full_memory_mib = estimated_full_memory_bytes / (1024 ** 2)
    absolute_difference = abs(
        estimated_full_memory_bytes - observed_memory_bytes
    )
    relative_percentage_error = (
        absolute_difference / observed_memory_bytes * 100
        if observed_memory_bytes != 0
        else 0.0
    )

    return {
        "average_memory_per_row_bytes": avg_memory_per_row,
        "estimated_full_memory_bytes": estimated_full_memory_bytes,
        "estimated_full_memory_mib": estimated_full_memory_mib,
        "absolute_difference_bytes": absolute_difference,
        "relative_percentage_error": relative_percentage_error,
    }


In [18]:
results = estimate_memory(
    sample_rows=100000,
    sample_memory_bytes=42808637,
    full_dataset_rows=12476012,
    observed_memory_bytes=5387774219
)

print(results)

{'average_memory_per_row_bytes': 428.08637, 'estimated_full_memory_bytes': 5340810689.15644, 'estimated_full_memory_mib': 5093.393983036461, 'absolute_difference_bytes': 46963529.84356022, 'relative_percentage_error': 0.8716684837672523}


In [13]:
#B1.3.3 - Column-Level Cardinality and Memory Assessment
import pandas as pd 
def cardinality_inspection(path_csv):
    try:
        df=pd.read_csv(path_csv,dtype=str,nrows=100_000,keep_default_na=False)
        #handle empty sample
        if df.empty:
            print("No records find in the CSV file.")
            return
        sample_rows=len(df)
        results=[]
        for column in df.columns:
            observed_data_type=df[column].dtype
            distinct_values=df[column].nunique()
            unique_ratio=distinct_values/sample_rows
            memory_bytes=df[column].memory_usage(deep=True,index=False)
            results.append({
                'column_name':column,
                'observed_data_type':observed_data_type,
                'distinct_values':distinct_values,
                'unique_ratio':unique_ratio,
                'memory_bytes':memory_bytes
            })
        profile_df = pd.DataFrame(results)

        print("\n=== COLUMN CARDINALITY PROFILE ===")
        

        return profile_df

    except FileNotFoundError:
        print(f"File not found")
    
    except pd.errors.EmptyDataError:
        print("CSV file is empty.")

    except Exception as e:
        print(f"Error: {e}")


In [16]:
#cardinality inspection transaction.csv
path_csv=r'C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv'
cardinality_inspection(path_csv)


=== COLUMN CARDINALITY PROFILE ===


,column_name,observed_data_type,distinct_values,unique_ratio,memory_bytes
0,TX_ID,object,100000,1.00000,5388895
1,SENDER_ACCOUNT_ID,object,31489,0.31489,5388249
2,RECEIVER_ACCOUNT_ID,object,50501,0.50501,5388601
3,TX_TYPE,object,1,0.00001,5700000
4,TX_AMOUNT,object,19766,0.19766,5442638
5,TIMESTAMP,object,2,0.00002,5000000
6,IS_FRAUD,object,2,0.00002,5399856
7,ALERT_ID,object,122,0.00122,5100266


In [18]:
#cardinality inspection accounts.csv
path_csv=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\accounts.csv"
cardinality_inspection(path_csv)


=== COLUMN CARDINALITY PROFILE ===


,column_name,observed_data_type,distinct_values,unique_ratio,memory_bytes
0,ACCOUNT_ID,object,100000,1.00000,5388890
1,CUSTOMER_ID,object,100000,1.00000,5588890
2,INIT_BALANCE,object,39706,0.39706,5500000
3,COUNTRY,object,1,0.00001,5100000
4,ACCOUNT_TYPE,object,1,0.00001,5000000
5,IS_FRAUD,object,2,0.00002,5382909
6,TX_BEHAVIOR_ID,object,5,0.00005,5000000


In [19]:
#cardinality inspection alerts.csv
path_csv=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\alerts.csv"
cardinality_inspection(path_csv)


=== COLUMN CARDINALITY PROFILE ===


,column_name,observed_data_type,distinct_values,unique_ratio,memory_bytes
0,ALERT_ID,object,3910,0.229299,899626
1,ALERT_TYPE,object,2,0.000117,928347
2,IS_FRAUD,object,1,0.000059,903756
3,TX_ID,object,17052,1.000000,956895
4,SENDER_ACCOUNT_ID,object,14923,0.875147,918945
5,RECEIVER_ACCOUNT_ID,object,10391,0.609371,918941
6,TX_TYPE,object,1,0.000059,971964
7,TX_AMOUNT,object,956,0.056064,911436
8,TIMESTAMP,object,200,0.011729,877257


In [ ]:
#B1.4 — Chunk-Based Transaction Ingestion
#B1.4.1 — Controlled chunk iteration
'''Technical Objective:Implement a bounded chunk-reading experiment that 
processes a small number of consecutive transaction batches.
'''
import pandas as pd
def process_chunk(csv_path):
    try:
        chunk_reader=pd.read_csv(
            csv_path,
            chunksize=50_000,
            dtype=str,
            keep_default_na=False
        )
        chunk_count=0
        cumulative_records=0
        for chunk in chunk_reader:
             chunk_count+=1
             rows=len(chunk)
             columns=len(chunk.columns)
             memory_bytes=(chunk.memory_usage(deep=True,index=False).sum())
             cumulative_records+=rows
             print('\n===Chunk Count===:',chunk_count)
             print('Rows:', rows)
             print('Columns:',columns)
             print('Mmeory Bytes:',memory_bytes)
             print('Cumulative Records:',cumulative_records)
             if chunk_count==3:
                break
        print("\n=== SUMMARY ===")
        print(
            f"Chunks Processed: "
            f"{chunk_count}"
        )
        print(
            f"Total Records Processed: "
            f"{cumulative_records:,}"
        )    
    except FileNotFoundError:
         print("File not found")
    except pd.errors.EmptyDataError:
            print("CSV file is empty.")
    except Exception as e:
            print(f"Error: {e}")
    
           

In [14]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
process_chunk(csv_path)

Chunk Count: 1
Rows: 50000
Columns: 8
Mmeory Bytes: 21397065
Cumulative Records: 50000
Chunk Count: 2
Rows: 50000
Columns: 8
Mmeory Bytes: 21411440
Cumulative Records: 100000
Chunk Count: 3
Rows: 50000
Columns: 8
Mmeory Bytes: 21461139
Cumulative Records: 150000

=== SUMMARY ===
Chunks Processed: 3
Total Records Processed: 150,000


In [39]:
#B1.4.2 — Full-File Streaming and Record Reconciliation
'''Technical objective:A Python function that iterates over the complete transactions.csv 
using fixed-size chunks and produces a final ingestion summary.
'''
import pandas as pd


def verify_full_streaming(csv_path):
    EXPECTED_RECORDS = 12_476_012

    try:
        total_chunks = 0
        cumulative_records = 0
        final_chunk_rows = 0
        max_chunk_memory = 0

        with pd.read_csv(
            csv_path,
            chunksize=50_000,
            dtype=str,
            keep_default_na=False
        ) as reader:

            for chunk in reader:

                total_chunks += 1

                rows = len(chunk)

                cumulative_records += rows

                final_chunk_rows = rows

                chunk_memory = (
                    chunk.memory_usage(deep=True)
                    .sum()
                )

                if chunk_memory > max_chunk_memory:
                    max_chunk_memory = chunk_memory

        print("\n=== STREAMING SUMMARY ===")
        print(f"Total Chunks Processed: {total_chunks}")
        print(f"Total Records Processed: {cumulative_records:,}")
        print(f"Final Chunk Rows: {final_chunk_rows:,}")
        print(
            f"Maximum Chunk Memory (bytes): "
            f"{max_chunk_memory:,}"
        )

        print("\n=== RECONCILIATION ===")

        if cumulative_records == EXPECTED_RECORDS:
            print(
                "Record count matches expected "
                "12,476,012 records."
            )
        else:
            print(
                "Record count mismatch."
            )
            print(
                f"Expected: {EXPECTED_RECORDS:,}"
            )
            print(
                f"Observed: {cumulative_records:,}"
            )

    except FileNotFoundError:
        print(f"File not found: {csv_path}")

    except pd.errors.EmptyDataError:
        print("CSV file is empty.")

    except Exception as e:
        print(
        f"Failure while processing "
        f"chunk {chunk_number}"
        )
        raise

In [19]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
verify_full_streaming(csv_path)


=== STREAMING SUMMARY ===
Total Chunks Processed: 250
Total Records Processed: 12,476,012
Final Chunk Rows: 26,012
Maximum Chunk Memory (bytes): 21,668,618

=== RECONCILIATION ===
Record count matches expected 12,476,012 records.


In [6]:
#B1.5 - Provenance, Logging and Failure Handling
#B1.5.1 — Implement fail-closed streaming verification
import pandas as pd


def verify_full_streaming(csv_path, expected_records):
    """
    Stream a CSV file in chunks and verify reconciliation.

    Returns:
        dict: Structured metrics on success.

    Raises:
        ValueError: Invalid input, zero records, or reconciliation failure.
        FileNotFoundError: Missing input file.
        pd.errors.EmptyDataError: Empty CSV.
        Any parsing exception from pandas.
    """

    if expected_records < 0:
        raise ValueError(
            "expected_records cannot be negative."
        )

    total_chunks = 0
    cumulative_records = 0
    final_chunk_rows = 0
    max_chunk_memory = 0

    with pd.read_csv(
        csv_path,
        chunksize=50_000,
        dtype=str,
        keep_default_na=False
    ) as reader:

        for chunk in reader:

            total_chunks += 1

            rows = len(chunk)

            cumulative_records += rows

            final_chunk_rows = rows

            chunk_memory = int(
                chunk.memory_usage(
                    deep=True
                ).sum()
            )

            max_chunk_memory = max(
                max_chunk_memory,
                chunk_memory
            )

    # Explicitly reject zero-record inputs
    if cumulative_records == 0:
        raise ValueError(
            "Zero transaction records processed."
        )

    reconciliation_status = (
        cumulative_records == expected_records
    )

    if not reconciliation_status:
        raise ValueError(
            f"Reconciliation failed. "
            f"Expected {expected_records:,} records but "
            f"processed {cumulative_records:,} records."
        )

    return {
        "total_chunks": total_chunks,
        "processed_rows": cumulative_records,
        "final_chunk_rows": final_chunk_rows,
        "max_chunk_memory_bytes": int(max_chunk_memory),
        "reconciliation_status": reconciliation_status
    }


In [25]:
csv_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
expected_records=12476012
verify_full_streaming(csv_path,expected_records)

{'total_chunks': 250,
 'processed_rows': 12476012,
 'final_chunk_rows': 26012,
 'max_chunk_memory_bytes': 21668618,
 'reconciliation_status': True}

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

# B1.5.1 Final Test Harness


with TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    success_csv = temp_dir / "success.csv"
    header_only_csv = temp_dir / "header_only.csv"
    missing_csv = temp_dir / "missing.csv"

    # -----------------------------
    # Create test fixtures
    # -----------------------------

    success_csv.write_text(
        "TX_ID,AMOUNT\n"
        "T1,100\n"
        "T2,200\n"
        "T3,300\n",
        encoding="utf-8"
    )

    header_only_csv.write_text(
        "TX_ID,AMOUNT\n",
        encoding="utf-8"
    )

    
    # T1 - Successful ingestion
   

    result = verify_full_streaming(
        success_csv,
        expected_records=3
    )

    assert result["processed_rows"] == 3
    assert result["total_chunks"] == 1
    assert result["final_chunk_rows"] == 3
    assert result["reconciliation_status"] is True
    assert isinstance(
        result["max_chunk_memory_bytes"],
        int
    )

    print("T1 PASS")

    
    # T2 - Reconciliation failure
   

    try:
        verify_full_streaming(
            success_csv,
            expected_records=5
        )

    except ValueError as e:
        assert "Reconciliation failed" in str(e)
        print("T2 PASS")

    else:
        raise AssertionError(
            "Expected ValueError for "
            "reconciliation mismatch"
        )

    
    # T3 - Missing file
   

    try:
        verify_full_streaming(
            missing_csv,
            expected_records=1
        )

    except FileNotFoundError:
        print("T3 PASS")

    else:
        raise AssertionError(
            "Expected FileNotFoundError"
        )

    
    # T4 - Header-only file
    

    try:
        verify_full_streaming(
            header_only_csv,
            expected_records=0
        )

    except ValueError as e:
        assert (
            "Zero transaction records processed"
            in str(e)
        )
        print("T4 PASS")

    else:
        raise AssertionError(
            "Expected ValueError for "
            "header-only file"
        )

print("\nAll B1.5.1 tests passed.")

T1 PASS
T2 PASS
T3 PASS
T4 PASS

All B1.5.1 tests passed.


In [4]:
#B1.5.2 — Source Provenance and Run Metadata
from pathlib import Path
from datetime import datetime,timezone
def extract_source_metadata(csv_path:Path,dataset_root:Path) -> dict:
    #preserve original paths
    csv_path=Path(csv_path)
    dataset_root=Path(dataset_root)
    #Reject symbolic links
    try:
        csv_path.lstat()
    except FileNotFoundError:
        raise FileNotFoundError(f"Source file not found:{csv_path}")
    if csv_path.is_symlink():
        raise ValueError(f"Symbolic-link sources are not allowed:{csv_path}")
    #resolve paths
    csv_path=csv_path.resolve()
    dataset_root=dataset_root.resolve()
    #validate dataset root 
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found:{dataset_root}")
    if not dataset_root.is_dir():
        raise NotADirectoryError(f"Dataset root is not a directory:{dataset_root}")
    #validate source file
    if not csv_path.exists():
        raise FileNotFoundError(f"Source File not found:{csv_path}")
    if not csv_path.is_file():
        raise ValueError(f"Source path is not a file:{csv_path}")
    
    try:
        relative_path=(
            csv_path.relative_to(dataset_root).as_posix()
        )
    except ValueError:
        raise ValueError("Source file must be inside dataset_root")
    #retriving filesystem metadata
    file_stat=csv_path.stat()
    file_size_bytes=file_stat.st_size
    last_modified_utc=datetime.fromtimestamp(file_stat.st_mtime,tz=timezone.utc
                                             ).isoformat(timespec="microseconds")  

    collected_at_utc=datetime.now(timezone.utc).isoformat(timespec="microseconds") 
    return {
        "source_file": csv_path.name,
        "relative_path": relative_path,
        "file_size_bytes": file_size_bytes,
        "last_modified_utc": last_modified_utc,
        "collected_at_utc": collected_at_utc
    }




In [31]:
DATASET_ROOT = Path(
    r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges"
)

TRANSACTIONS_FILE = (
    DATASET_ROOT / "transactions.csv"
)

metadata = extract_source_metadata(
    TRANSACTIONS_FILE,
    DATASET_ROOT
)

print(metadata)

{'source_file': 'transactions.csv', 'relative_path': 'transactions.csv', 'file_size_bytes': 609461582, 'last_modified_utc': '2019-06-04T02:21:13.000000+00:00', 'collected_at_utc': '2026-11-24T16:34:27.754479+00:00'}


In [9]:
from pathlib import Path
from tempfile import TemporaryDirectory
from unittest.mock import patch

# -------------------------------------------------
# B1.5.2.1 Final Verification Test Harness
# Assumes extract_source_metadata() already exists.
# -------------------------------------------------

with TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    dataset_root = temp_dir / "dataset"
    dataset_root.mkdir()

    # -----------------------------------------
    # Create test fixtures
    # -----------------------------------------

    normal_csv = dataset_root / "transactions.csv"

    normal_csv.write_text(
        "TX_ID,AMOUNT\n"
        "T1,100\n",
        encoding="utf-8"
    )

    outside_csv = temp_dir / "outside.csv"

    outside_csv.write_text(
        "TX_ID,AMOUNT\n"
        "T2,200\n",
        encoding="utf-8"
    )

    # -----------------------------------------
    # T1 - Normal CSV inside dataset root
    # Expected: metadata returned
    # -----------------------------------------

    metadata = extract_source_metadata(
        normal_csv,
        dataset_root
    )

    expected_size = normal_csv.stat().st_size

    required_fields = {
        "source_file",
        "relative_path",
        "file_size_bytes",
        "last_modified_utc",
        "collected_at_utc"
    }

    assert required_fields.issubset(metadata.keys())

    assert metadata["source_file"] == "transactions.csv"
    assert metadata["relative_path"] == "transactions.csv"
    assert metadata["file_size_bytes"] == expected_size

    # Verify UTC-aware timestamps
    assert "+00:00" in metadata["last_modified_utc"]
    assert "+00:00" in metadata["collected_at_utc"]

    print("T1 PASS")

    # -----------------------------------------
    # T2 - Real symbolic link (if permitted)
    # Expected: ValueError
    # -----------------------------------------

    symlink_path = dataset_root / "transactions_link.csv"

    try:
        symlink_path.symlink_to(normal_csv)

    except (OSError, NotImplementedError) as e:
        print(
            f"T2 SKIPPED - Unable to create symbolic link: {e}"
        )

    else:
        try:
            extract_source_metadata(
                symlink_path,
                dataset_root
            )

        except ValueError as e:
            assert (
                "symbolic" in str(e).lower()
                or
                "symlink" in str(e).lower()
            )

            print("T2 PASS")

        else:
            raise AssertionError(
                "Expected ValueError for symbolic-link source"
            )

    # -----------------------------------------
    # T3 - File outside dataset root
    # Expected: ValueError
    # -----------------------------------------

    try:
        extract_source_metadata(
            outside_csv,
            dataset_root
        )

    except ValueError as e:
        assert "dataset_root" in str(e)

        print("T3 PASS")

    else:
        raise AssertionError(
            "Expected ValueError for source outside dataset root"
        )

    # -----------------------------------------
    # T4 - Simulated symbolic-link detection
    # Expected: ValueError
    # Uses unittest.mock instead of a real symlink
    # -----------------------------------------

    with patch.object(
        Path,
        "is_symlink",
        return_value=True
    ):

        try:
            extract_source_metadata(
                normal_csv,
                dataset_root
            )

        except ValueError as e:

            assert (
                "symbolic" in str(e).lower()
                or
                "symlink" in str(e).lower()
            )

            print("T4 PASS")

        else:
            raise AssertionError(
                "Expected ValueError when "
                "Path.is_symlink() returns True"
            )

print("\nAll B1.5.2.1 verification tests passed.")

T1 PASS
T2 SKIPPED - Unable to create symbolic link: [WinError 1314] A required privilege is not held by the client: 'C:\\Users\\maiti\\AppData\\Local\\Temp\\tmp7cte6tnv\\dataset\\transactions.csv' -> 'C:\\Users\\maiti\\AppData\\Local\\Temp\\tmp7cte6tnv\\dataset\\transactions_link.csv'
T3 PASS
T4 PASS

All B1.5.2.1 verification tests passed.


In [5]:
#B1.5.2.2 — SHA-256 fingerprint
from pathlib import Path
import hashlib
block_size=1_048_576
def calculate_sha256(file_path:Path) -> str:
 """
     Calculate the SHA-256 fingerprint of a file using
     fixed-size block reads.
 
     Parameters
     ----------
     file_path : Path
         Source file path.
 
     Returns
     -------
     str
         64-character hexadecimal SHA-256 digest.
 
     Raises
     ------
     FileNotFoundError
         If the file does not exist.
     OSError
         If the file cannot be read.
     """
 file_path=Path(file_path)
 if not file_path.exists():
  raise FileNotFoundError(f"file not found:{file_path}")
 if not file_path.is_file():
  raise ValueError(f"Not a file:{file_path}")
 sha256_hash=hashlib.sha256()
 with file_path.open('rb') as file:
  while True:
   block=file.read(block_size)
   if not block:
     break
   sha256_hash.update(block)
 return sha256_hash.hexdigest()


In [7]:
file_path=r"C:\Users\maiti\CAREER_HQ\09_PROJECTS_PORTFOLIO\Coordinated Fraud Ring Detection and Investigation\dataset\raw\100Kvertices-10Medges\transactions.csv"
calculate_sha256(file_path)

'b27bcbe175411870eccacd637e173d44ae453845605dc540b76916e5da625834'

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
# B1.5.2.2 SHA-256 Verification Tests
# Assumes calculate_sha256() already exists
with TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)


    # T1 - Known Answer Test ("abc")


    abc_file = temp_dir / "abc.txt"
    abc_file.write_bytes(b"abc")

    digest = calculate_sha256(abc_file)

    expected_digest = (
        "ba7816bf8f01cfea414140de5dae2223"
        "b00361a396177a9cb410ff61f20015ad"
    )

    assert digest == expected_digest

    print("T1 PASS")

    
    # T2 - Repeatability
    digest1 = calculate_sha256(abc_file)
    digest2 = calculate_sha256(abc_file)

    assert digest1 == digest2

    print("T2 PASS")

    # T3 - Content Modification 

    original_digest = calculate_sha256(abc_file)

    abc_file.write_bytes(b"abd")

    modified_digest = calculate_sha256(abc_file)

    assert original_digest != modified_digest

    print("T3 PASS")

    # T4 - Empty File

    empty_file = temp_dir / "empty.txt"
    empty_file.write_bytes(b"")

    empty_digest = calculate_sha256(empty_file)

    expected_empty_digest = (
        "e3b0c44298fc1c149afbf4c8996fb924"
        "27ae41e4649b934ca495991b7852b855"
    )

    assert empty_digest == expected_empty_digest

    print("T4 PASS")

    
    # T5 - Missing File

    missing_file = temp_dir / "missing.txt"

    try:
        calculate_sha256(missing_file)

    except FileNotFoundError:
        print("T5 PASS")

    else:
        raise AssertionError(
            "Expected FileNotFoundError"
        )

    
    # T6 - Multiple Blocks (> 1 MiB)

    large_file = temp_dir / "large.bin"

    large_content = b"A" * (1_048_576 + 500)

    large_file.write_bytes(large_content)

    production_digest = calculate_sha256(
        large_file
    )

    reference_digest = hashlib.sha256(
        large_content
    ).hexdigest()

    assert production_digest == reference_digest

    print("T6 PASS")

print("\nAll B1.5.2.2 verification tests passed.")

T1 PASS
T2 PASS
T3 PASS
T4 PASS
T5 PASS
T6 PASS

All B1.5.2.2 verification tests passed.


In [8]:
#B1.5.2.3.1 — Unified provenance record
from uuid import uuid4
from datetime import datetime, timezone
import json
import string


def build_run_provenance(
    metadata: dict,
    sha256_digest: str,
    streaming_result: dict
) -> dict:
    """
    Build a unified provenance record from:
    - metadata dictionary
    - SHA-256 fingerprint
    - streaming verification result

    Returns a JSON-serializable dictionary.
    """

    # ----------------------------------
    # Validate metadata dictionary
    # ----------------------------------

    required_metadata_keys = {
        "source_file",
        "relative_path",
        "file_size_bytes",
        "last_modified_utc"
    }

    missing_metadata = (
        required_metadata_keys
        - set(metadata.keys())
    )

    if missing_metadata:
        raise ValueError(
            f"Missing metadata fields: "
            f"{sorted(missing_metadata)}"
        )

    # ----------------------------------
    # Validate SHA-256 digest
    # ----------------------------------

    if not isinstance(sha256_digest, str):
        raise ValueError(
            "SHA-256 digest must be a string."
        )

    HEX_CHARS = set(string.hexdigits)

    if (
        len(sha256_digest) != 64
        or not all(
            character in HEX_CHARS
            for character in sha256_digest
        )
    ):
        raise ValueError(
            "SHA-256 digest must contain exactly "
            "64 hexadecimal characters."
        )

    # ----------------------------------
    # Validate streaming result keys
    # ----------------------------------

    required_streaming_keys = {
        "processed_rows",
        "total_chunks",
        "reconciliation_status"
    }

    missing_streaming = (
        required_streaming_keys
        - set(streaming_result.keys())
    )

    if missing_streaming:
        raise ValueError(
            f"Missing streaming fields: "
            f"{sorted(missing_streaming)}"
        )

    # ----------------------------------
    # Validate reconciliation status
    # Must be the actual Boolean True
    # ----------------------------------

    reconciliation_status = (
        streaming_result["reconciliation_status"]
    )

    if reconciliation_status is not True:
        raise ValueError(
            "reconciliation_status must be "
            "the Boolean True."
        )

    # ----------------------------------
    # Validate processed_rows
    # ----------------------------------

    processed_rows = (
        streaming_result["processed_rows"]
    )

    if (
        isinstance(processed_rows, bool)
        or not isinstance(processed_rows, int)
        or processed_rows <= 0
    ):
        raise ValueError(
            "processed_rows must be a "
            "positive integer."
        )

    # ----------------------------------
    # Validate total_chunks
    # ----------------------------------

    total_chunks = (
        streaming_result["total_chunks"]
    )

    if (
        isinstance(total_chunks, bool)
        or not isinstance(total_chunks, int)
        or total_chunks <= 0
    ):
        raise ValueError(
            "total_chunks must be a "
            "positive integer."
        )

    # ----------------------------------
    # Build provenance record
    # ----------------------------------

    provenance_record = {
        "run_id": str(uuid4()),

        "record_created_at_utc": (
            datetime.now(timezone.utc)
            .isoformat(timespec="microseconds")
        ),

        "source_file":
            metadata["source_file"],

        "relative_path":
            metadata["relative_path"],

        "file_size_bytes":
            int(metadata["file_size_bytes"]),

        "last_modified_utc":
            metadata["last_modified_utc"],

        "source_sha256":
            sha256_digest,

        "records_processed":
            processed_rows,

        "total_chunks":
            total_chunks,

        "reconciliation_status":
            reconciliation_status,

        "run_status":
            "success"
    }

    # ----------------------------------
    # Verify JSON serializability
    # ----------------------------------

    json.dumps(provenance_record)

    return provenance_record

In [14]:
metadata = {
    "source_file": "transactions.csv",
    "relative_path": "transactions.csv",
    "file_size_bytes": 845672341,
    "last_modified_utc": "2026-11-12T08:00:00+00:00",
    "collected_at_utc": "2026-11-12T08:05:00+00:00"
}

sha256_digest = (
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

streaming_result = {
    "processed_rows": 12476012,
    "total_chunks": 250,
    "reconciliation_status": True
}

provenance = build_run_provenance(
    metadata,
    sha256_digest,
    streaming_result
)

print(provenance)

{'run_id': '12c888bb-ee8d-45ce-b86e-2ea024d8fc4e', 'record_created_at_utc': '2026-11-23T10:27:00.707885+00:00', 'source_file': 'transactions.csv', 'relative_path': 'transactions.csv', 'file_size_bytes': 845672341, 'last_modified_utc': '2026-11-12T08:00:00+00:00', 'source_sha256': 'ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad', 'records_processed': 12476012, 'total_chunks': 250, 'reconciliation_status': True, 'run_status': 'success'}


In [15]:
#test harness for B1.5.2.3.1 — Unified provenance record
import json

# -----------------------------
# Valid test fixtures
# -----------------------------

valid_metadata = {
    "source_file": "transactions.csv",
    "relative_path": "transactions.csv",
    "file_size_bytes": 100,
    "last_modified_utc": "2026-11-12T10:00:00+00:00"
}

valid_sha256 = (
    "a" * 64
)

valid_streaming_result = {
    "processed_rows": 100,
    "total_chunks": 2,
    "reconciliation_status": True
}

# -----------------------------
# T1 - All valid inputs
# Expected: success
# -----------------------------

record = build_run_provenance(
    valid_metadata,
    valid_sha256,
    valid_streaming_result
)

assert record["run_status"] == "success"
assert record["reconciliation_status"] is True

# Verify JSON serializable
json.dumps(record)

print("T1 PASS")

# -----------------------------
# T2 - reconciliation_status=False
# Expected: ValueError
# -----------------------------

try:
    bad_streaming = valid_streaming_result.copy()
    bad_streaming["reconciliation_status"] = False

    build_run_provenance(
        valid_metadata,
        valid_sha256,
        bad_streaming
    )

except ValueError:
    print("T2 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "reconciliation_status=False"
    )

# -----------------------------
# T3 - reconciliation_status='False'
# Expected: ValueError
# -----------------------------

try:
    bad_streaming = valid_streaming_result.copy()
    bad_streaming["reconciliation_status"] = "False"

    build_run_provenance(
        valid_metadata,
        valid_sha256,
        bad_streaming
    )

except ValueError:
    print("T3 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "reconciliation_status='False'"
    )

# -----------------------------
# T4 - processed_rows=-10
# Expected: ValueError
# -----------------------------

try:
    bad_streaming = valid_streaming_result.copy()
    bad_streaming["processed_rows"] = -10

    build_run_provenance(
        valid_metadata,
        valid_sha256,
        bad_streaming
    )

except ValueError:
    print("T4 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "negative processed_rows"
    )

# -----------------------------
# T5 - total_chunks=0
# Expected: ValueError
# -----------------------------

try:
    bad_streaming = valid_streaming_result.copy()
    bad_streaming["total_chunks"] = 0

    build_run_provenance(
        valid_metadata,
        valid_sha256,
        bad_streaming
    )

except ValueError:
    print("T5 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "total_chunks=0"
    )

# -----------------------------
# T6 - processed_rows=12.5
# Expected: ValueError
# -----------------------------

try:
    bad_streaming = valid_streaming_result.copy()
    bad_streaming["processed_rows"] = 12.5

    build_run_provenance(
        valid_metadata,
        valid_sha256,
        bad_streaming
    )

except ValueError:
    print("T6 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "fractional processed_rows"
    )

# -----------------------------
# T7 - SHA-256 containing _
# Expected: ValueError
# -----------------------------

try:
    bad_sha256 = (
        "a" * 63 + "_"
    )

    build_run_provenance(
        valid_metadata,
        bad_sha256,
        valid_streaming_result
    )

except ValueError:
    print("T7 PASS")

else:
    raise AssertionError(
        "Expected ValueError for "
        "invalid SHA-256 digest"
    )

print("\nAll B1.5.2.3.1 tests passed.")

T1 PASS
T2 PASS
T3 PASS
T4 PASS
T5 PASS
T6 PASS
T7 PASS

All B1.5.2.3.1 tests passed.


In [9]:
#B1.5.3.1 — Append a successful ingestion event to JSONL
from pathlib import Path
from datetime import datetime, timezone
import json


def append_ingestion_log(
    provenance_record: dict,
    log_path: Path
) -> None:

    # Require successful reconciliation

    if (
        provenance_record.get("run_status") != "success"
        or provenance_record.get("reconciliation_status")
        is not True
    ):
        raise ValueError(
            "Only successfully reconciled "
            "ingestion records may be logged."
        )

    run_id=provenance_record.get("run_id")
    if not isinstance(run_id,str) or not run_id.strip():
        raise ValueError("A valid run_id is required")
    event = {
        "event_type": "ingestion_success",

        "logged_at_utc": (
            datetime.now(timezone.utc)
            .isoformat(timespec="microseconds")
        ),

        "run_id":
            provenance_record["run_id"],

        "run_status":
            "success",

        "provenance":
            provenance_record
    }

    # Serialize BEFORE opening the file
    event_json = json.dumps(event)

    log_path = Path(log_path)

    with log_path.open(
        mode="a",
        encoding="utf-8"
    ) as log_file:

        # Single write operation
        log_file.write(event_json + "\n")

In [12]:
provenance_record = {
    "run_id": "123",
    "record_created_at_utc": "2026-11-12T10:00:00+00:00",
    "source_file": "transactions.csv",
    "relative_path": "transactions.csv",
    "file_size_bytes": 100,
    "last_modified_utc": "2026-11-12T09:00:00+00:00",
    "source_sha256": "a" * 64,
    "records_processed": 100,
    "total_chunks": 2,
    "reconciliation_status": True,
    "run_status": "success"
}
append_ingestion_log(
    provenance_record,
    "ingestion_log.jsonl"
)

In [13]:
#Test harness for B1.5.3.1 — Append a successful ingestion event to JSONL
from pathlib import Path
from tempfile import TemporaryDirectory
import json

with TemporaryDirectory() as temp_dir:

    log_path = Path(temp_dir) / "ingestion_log.jsonl"

    # ----------------------------------
    # T1 - Append one valid success event
    # ----------------------------------

    record_1 = {
        "run_id": "run-001",
        "run_status": "success",
        "reconciliation_status": True,
        "records_processed": 100,
        "total_chunks": 2
    }

    append_ingestion_log(record_1, log_path)

    lines = log_path.read_text(
        encoding="utf-8"
    ).splitlines()

    assert len(lines) == 1

    event_1 = json.loads(lines[0])

    assert event_1["event_type"] == "ingestion_success"
    assert event_1["run_id"] == "run-001"

    print("T1 PASS")

    # ----------------------------------
    # T2 - Append second valid event
    # ----------------------------------

    record_2 = {
        "run_id": "run-002",
        "run_status": "success",
        "reconciliation_status": True,
        "records_processed": 200,
        "total_chunks": 4
    }

    append_ingestion_log(record_2, log_path)

    lines = log_path.read_text(
        encoding="utf-8"
    ).splitlines()

    assert len(lines) == 2

    first_event = json.loads(lines[0])
    second_event = json.loads(lines[1])

    assert first_event["run_id"] == "run-001"
    assert second_event["run_id"] == "run-002"

    print("T2 PASS")

    # ----------------------------------
    # T3 - run_status = failed
    # Log must remain unchanged
    # ----------------------------------

    before = log_path.read_text(
        encoding="utf-8"
    )

    bad_record = record_1.copy()
    bad_record["run_status"] = "failed"

    try:
        append_ingestion_log(
            bad_record,
            log_path
        )

    except ValueError:
        after = log_path.read_text(
            encoding="utf-8"
        )

        assert before == after
        print("T3 PASS")

    else:
        raise AssertionError(
            "Expected ValueError"
        )

    # ----------------------------------
    # T4 - reconciliation_status=False
    # Log must remain unchanged
    # ----------------------------------

    before = log_path.read_text(
        encoding="utf-8"
    )

    bad_record = record_1.copy()
    bad_record["reconciliation_status"] = False

    try:
        append_ingestion_log(
            bad_record,
            log_path
        )

    except ValueError:
        after = log_path.read_text(
            encoding="utf-8"
        )

        assert before == after
        print("T4 PASS")

    else:
        raise AssertionError(
            "Expected ValueError"
        )

    # ----------------------------------
    # T5 - Non-serializable data
    # Log must remain unchanged
    # ----------------------------------

    before = log_path.read_text(
        encoding="utf-8"
    )

    bad_record = record_1.copy()

    # set() is not JSON serializable
    bad_record["bad_value"] = {1, 2, 3}

    try:
        append_ingestion_log(
            bad_record,
            log_path
        )

    except TypeError:
        after = log_path.read_text(
            encoding="utf-8"
        )

        assert before == after
        print("T5 PASS")

    else:
        raise AssertionError(
            "Expected TypeError"
        )

    # ----------------------------------
    # T6 - Verify event fields preserved
    # ----------------------------------

    lines = log_path.read_text(
        encoding="utf-8"
    ).splitlines()

    first_event = json.loads(lines[0])

    assert first_event["event_type"] == (
        "ingestion_success"
    )

    assert first_event["run_status"] == (
        "success"
    )

    assert first_event["run_id"] == (
        "run-001"
    )

    assert (
        first_event["provenance"]["run_id"]
        == "run-001"
    )

    assert (
        first_event["provenance"][
            "records_processed"
        ] == 100
    )

    print("T6 PASS")

print("\nAll B1.5.3.1 tests passed.")

T1 PASS
T2 PASS
T3 PASS
T4 PASS
T5 PASS
T6 PASS

All B1.5.3.1 tests passed.


In [21]:
#B1.5.3.2 — Failed-Ingestion Execution Logging
from pathlib import Path
from datetime import datetime,timezone
import json
def append_failed_ingestion_log(
        run_id:str,
        failure_stage:str,
        error:Exception,
        log_path
)-> None:
    #validate run_id
    if (not isinstance(run_id,str) or not run_id.strip()):
        raise ValueError("run_id must be non empty string")
    #validate failure stage
    if (not isinstance(failure_stage,str) or not failure_stage.strip()):
        raise ValueError("failure_satge must be a non-empty string")
    #validate error object
    if not isinstance(error,Exception):
        raise ValueError("error must be an exception instance")
    #build failure event
    event={
        "event_type":"ingestion_failure",
        "logged_at_utc":(datetime.now(timezone.utc).isoformat(timespec="microseconds")),
        "run_id":run_id,
        "run_status":"failed",
        "failure_stage":failure_stage,
        "error_type":(error.__class__.__name__)
    }
    #serialize before opening file
    event_json=json.dumps(event)
    log_path=Path(log_path)
    #append one JSONL line
    with log_path.open(
        mode='a',
        encoding='utf-8'
    ) as log_file:
        log_file.write(
            event_json + "\n"
        )


In [15]:
try:
    raise FileNotFoundError(
        "transactions.csv not found"
    )

except Exception as e:

    append_failed_ingestion_log(
        run_id="run-001",
        failure_stage="streaming_verification",
        error=e,
        log_path="ingestion_log.jsonl"
    )

In [16]:
#test harness for B1.5.3.2 — Failed-Ingestion Execution Logging
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime, timezone
import json


def test_append_failed_ingestion_log() -> None:

    # Helper function to read JSONL records
    def read_records(path):
        return [
            json.loads(line)
            for line in path.read_text(
                encoding="utf-8"
            ).splitlines()
        ]

    with TemporaryDirectory() as temp_dir:

        log_path = Path(temp_dir) / "failure.jsonl"

        # ==========================================
        # T1 — Valid failure event
        # ==========================================

        try:
            raise FileNotFoundError(
                "transactions.csv not found"
            )

        except FileNotFoundError as e:

            append_failed_ingestion_log(
                run_id="run-001",
                failure_stage="streaming_verification",
                error=e,
                log_path=log_path
            )

        records = read_records(log_path)

        assert len(records) == 1

        record = records[0]

        # Validate exact event schema
        assert set(record) == {
            "event_type",
            "logged_at_utc",
            "run_id",
            "run_status",
            "failure_stage",
            "error_type"
        }

        # Validate field values
        assert record["event_type"] == "ingestion_failure"

        assert record["run_id"] == "run-001"

        assert record["run_status"] == "failed"

        assert record["failure_stage"] == (
            "streaming_verification"
        )

        assert record["error_type"] == "FileNotFoundError"

        # Validate UTC timestamp
        timestamp = datetime.fromisoformat(
            record["logged_at_utc"]
        )

        assert timestamp.tzinfo == timezone.utc

        # Validate newline termination
        assert log_path.read_bytes().endswith(b"\n")

        print("T1 PASSED: Valid failure event")


        # ==========================================
        # T2 — Success followed by failure event
        # ==========================================

        log_path = Path(temp_dir) / "mixed_events.jsonl"

        provenance = {
            "run_id": "run-002",
            "run_status": "success",
            "reconciliation_status": True
        }

        # Append successful ingestion event
        append_ingestion_log(
            provenance,
            log_path
        )

        # Capture existing log content
        before = log_path.read_bytes()

        # Append failed ingestion event
        append_failed_ingestion_log(
            run_id="run-003",
            failure_stage="schema_validation",
            error=ValueError("Invalid schema"),
            log_path=log_path
        )

        records = read_records(log_path)

        assert len(records) == 2

        # Validate success event
        assert records[0]["event_type"] == "ingestion_success"

        assert records[0]["provenance"] == provenance

        # Validate failure event
        assert records[1]["event_type"] == "ingestion_failure"

        assert records[1]["run_id"] == "run-003"

        assert records[1]["run_status"] == "failed"

        assert records[1]["failure_stage"] == "schema_validation"

        assert records[1]["error_type"] == "ValueError"

        # Verify existing record was not modified
        assert log_path.read_bytes().startswith(before)

        print("T2 PASSED: Both events preserved")


        # ==========================================
        # T3, T4, T5 — Invalid input validation
        # ==========================================

        invalid_cases = [

            # T3 — Blank run_id
            (
                "T3",
                "   ",
                "streaming_verification",
                ValueError("Test error")
            ),

            # T4 — Blank failure_stage
            (
                "T4",
                "run-004",
                "   ",
                ValueError("Test error")
            ),

            # T5 — String instead of exception
            (
                "T5",
                "run-005",
                "streaming_verification",
                "Not an exception"
            )
        ]

        for test_id, run_id, failure_stage, error in invalid_cases:

            # Capture log before invalid operation
            before = log_path.read_bytes()

            try:

                append_failed_ingestion_log(
                    run_id=run_id,
                    failure_stage=failure_stage,
                    error=error,
                    log_path=log_path
                )

            except ValueError:
                pass

            else:
                raise AssertionError(
                    f"{test_id} FAILED: Expected ValueError"
                )

            # Ensure complete file remains unchanged
            assert log_path.read_bytes() == before, (
                f"{test_id} FAILED: Log modified"
            )

            print(
                f"{test_id} PASSED: "
                "Invalid input rejected; log unchanged"
            )

    print("\nAll 5 test cases passed successfully.")

In [17]:
test_append_failed_ingestion_log()

T1 PASSED: Valid failure event
T2 PASSED: Both events preserved
T3 PASSED: Invalid input rejected; log unchanged
T4 PASSED: Invalid input rejected; log unchanged
T5 PASSED: Invalid input rejected; log unchanged

All 5 test cases passed successfully.


In [13]:
#B1.6 — Final Raw Ingestion Integration
#B1.6.1 - Execution Identity Consistency,
#Unified provenance with externally assigned run_id

from datetime import datetime, timezone
import json
import string


def build_run_provenance(
    metadata: dict,
    sha256_digest: str,
    streaming_result: dict,
    run_id: str
) -> dict:

    """
    Build a JSON-serializable provenance record using
    an externally assigned execution identifier.
    """

    # ----------------------------------
    # Validate run_id
    # ----------------------------------

    if (
        not isinstance(run_id, str)
        or not run_id.strip()
    ):
        raise ValueError(
            "run_id must be a non-empty string."
        )

    # ----------------------------------
    # Validate metadata dictionary
    # ----------------------------------

    required_metadata_keys = {
        "source_file",
        "relative_path",
        "file_size_bytes",
        "last_modified_utc"
    }

    missing_metadata = (
        required_metadata_keys
        - set(metadata.keys())
    )

    if missing_metadata:
        raise ValueError(
            f"Missing metadata fields: "
            f"{sorted(missing_metadata)}"
        )

    # ----------------------------------
    # Validate SHA-256 digest
    # ----------------------------------

    if not isinstance(sha256_digest, str):
        raise ValueError(
            "SHA-256 digest must be a string."
        )

    HEX_CHARS = set(string.hexdigits)

    if (
        len(sha256_digest) != 64
        or not all(
            character in HEX_CHARS
            for character in sha256_digest
        )
    ):
        raise ValueError(
            "SHA-256 digest must contain exactly "
            "64 hexadecimal characters."
        )

    # ----------------------------------
    # Validate streaming result keys
    # ----------------------------------

    required_streaming_keys = {
        "processed_rows",
        "total_chunks",
        "reconciliation_status"
    }

    missing_streaming = (
        required_streaming_keys
        - set(streaming_result.keys())
    )

    if missing_streaming:
        raise ValueError(
            f"Missing streaming fields: "
            f"{sorted(missing_streaming)}"
        )

    # ----------------------------------
    # Validate reconciliation status
    # ----------------------------------

    reconciliation_status = (
        streaming_result["reconciliation_status"]
    )

    if reconciliation_status is not True:
        raise ValueError(
            "reconciliation_status must be "
            "the Boolean True."
        )

    # ----------------------------------
    # Validate processed_rows
    # ----------------------------------

    processed_rows = streaming_result["processed_rows"]

    if (
        isinstance(processed_rows, bool)
        or not isinstance(processed_rows, int)
        or processed_rows <= 0
    ):
        raise ValueError(
            "processed_rows must be a positive integer."
        )

    # ----------------------------------
    # Validate total_chunks
    # ----------------------------------

    total_chunks = streaming_result["total_chunks"]

    if (
        isinstance(total_chunks, bool)
        or not isinstance(total_chunks, int)
        or total_chunks <= 0
    ):
        raise ValueError(
            "total_chunks must be a positive integer."
        )

    # ----------------------------------
    # Build provenance record
    # ----------------------------------

    provenance_record = {

        "run_id": run_id,

        "record_created_at_utc": (
            datetime.now(timezone.utc)
            .isoformat(timespec="microseconds")
        ),

        "source_file":
            metadata["source_file"],

        "relative_path":
            metadata["relative_path"],

        "file_size_bytes":
            int(metadata["file_size_bytes"]),

        "last_modified_utc":
            metadata["last_modified_utc"],

        "source_sha256":
            sha256_digest,

        "records_processed":
            processed_rows,

        "total_chunks":
            total_chunks,

        "reconciliation_status":
            reconciliation_status,

        "run_status":
            "success"
    }

    # ----------------------------------
    # Verify JSON serializability
    # ----------------------------------

    json.dumps(provenance_record)

    return provenance_record

In [28]:
from uuid import uuid4
import json

# Generate execution identity externally
run_id = str(uuid4())

# Sample metadata
metadata = {
    "source_file": "transactions.csv",
    "relative_path": "raw/transactions.csv",
    "file_size_bytes": 102400,
    "last_modified_utc": "2026-11-24T10:00:00+00:00"
}

# Sample SHA-256 fingerprint
sha256_digest = "a" * 64

# Sample streaming verification result
streaming_result = {
    "processed_rows": 10000,
    "total_chunks": 10,
    "reconciliation_status": True
}

# Execute the revised function
provenance_record = build_run_provenance(
    metadata=metadata,
    sha256_digest=sha256_digest,
    streaming_result=streaming_result,
    run_id=run_id
)

# Display provenance record
print(json.dumps(provenance_record, indent=4))

{
    "run_id": "724393b2-3b2d-4b53-9c98-a714d75f98b6",
    "record_created_at_utc": "2026-11-24T15:21:11.655298+00:00",
    "source_file": "transactions.csv",
    "relative_path": "raw/transactions.csv",
    "file_size_bytes": 102400,
    "last_modified_utc": "2026-11-24T10:00:00+00:00",
    "source_sha256": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
    "records_processed": 10000,
    "total_chunks": 10,
    "reconciliation_status": true,
    "run_status": "success"
}


In [33]:
metadata = extract_source_metadata(TRANSACTIONS_FILE, DATASET_ROOT)
from uuid import uuid4

run_id = str(uuid4())

provenance_record = build_run_provenance(
    metadata=metadata,
    sha256_digest=sha256_digest,
    streaming_result=streaming_result,
    run_id=run_id
)

print(json.dumps(provenance_record, indent=4))

{
    "run_id": "f453e7d1-2e2f-4675-9e52-a0c98811faf7",
    "record_created_at_utc": "2026-11-24T16:42:26.555647+00:00",
    "source_file": "transactions.csv",
    "relative_path": "transactions.csv",
    "file_size_bytes": 609461582,
    "last_modified_utc": "2019-06-04T02:21:13.000000+00:00",
    "source_sha256": "aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
    "records_processed": 10000,
    "total_chunks": 10,
    "reconciliation_status": true,
    "run_status": "success"
}


In [34]:
#Required tests for B1.6.1 verification
from uuid import uuid4


def test_build_run_provenance() -> None:

    # Synthetic metadata
    metadata = {
        "source_file": "transactions.csv",
        "relative_path": "raw/transactions.csv",
        "file_size_bytes": 102400,
        "last_modified_utc": "2026-11-24T10:00:00+00:00"
    }

    # Synthetic SHA-256 digest
    sha256_digest = "a" * 64

    # Synthetic streaming result
    streaming_result = {
        "processed_rows": 10000,
        "total_chunks": 10,
        "reconciliation_status": True
    }

    # ==========================================
    # T1 — Valid externally supplied UUID
    # ==========================================

    run_id = str(uuid4())

    record = build_run_provenance(
        metadata=metadata,
        sha256_digest=sha256_digest,
        streaming_result=streaming_result,
        run_id=run_id
    )

    assert record["run_id"] == run_id, (
        "T1 FAILED: run_id was not preserved"
    )

    print("T1 PASSED: Exact UUID preserved")


    # ==========================================
    # T2 — Whitespace-only identifier
    # ==========================================

    try:

        build_run_provenance(
            metadata=metadata,
            sha256_digest=sha256_digest,
            streaming_result=streaming_result,
            run_id="   "
        )

    except ValueError:

        print("T2 PASSED: Whitespace-only run_id rejected")

    else:

        raise AssertionError(
            "T2 FAILED: Expected ValueError"
        )


    # ==========================================
    # T3 — Integer identifier
    # ==========================================

    try:

        build_run_provenance(
            metadata=metadata,
            sha256_digest=sha256_digest,
            streaming_result=streaming_result,
            run_id=12345
        )

    except ValueError:

        print("T3 PASSED: Integer run_id rejected")

    else:

        raise AssertionError(
            "T3 FAILED: Expected ValueError"
        )


    # ==========================================
    # T4 — Same identifier used in two calls
    # ==========================================

    shared_run_id = str(uuid4())

    record_1 = build_run_provenance(
        metadata=metadata,
        sha256_digest=sha256_digest,
        streaming_result=streaming_result,
        run_id=shared_run_id
    )

    record_2 = build_run_provenance(
        metadata=metadata,
        sha256_digest=sha256_digest,
        streaming_result=streaming_result,
        run_id=shared_run_id
    )

    assert (
        record_1["run_id"]
        == record_2["run_id"]
        == shared_run_id
    ), "T4 FAILED: Inconsistent run identity"

    print("T4 PASSED: Both records retain the same run_id")


    # ==========================================
    # T5 — Failed reconciliation with valid run_id
    # ==========================================

    failed_streaming_result = streaming_result.copy()

    failed_streaming_result["reconciliation_status"] = False

    try:

        build_run_provenance(
            metadata=metadata,
            sha256_digest=sha256_digest,
            streaming_result=failed_streaming_result,
            run_id=str(uuid4())
        )

    except ValueError:

        print("T5 PASSED: Failed reconciliation rejected")

    else:

        raise AssertionError(
            "T5 FAILED: Failed reconciliation accepted"
        )


    print("\nAll 5 test cases passed successfully.")


In [35]:
test_build_run_provenance()

T1 PASSED: Exact UUID preserved
T2 PASSED: Whitespace-only run_id rejected
T3 PASSED: Integer run_id rejected
T4 PASSED: Both records retain the same run_id
T5 PASSED: Failed reconciliation rejected

All 5 test cases passed successfully.


In [14]:
#B1.6.2 — End-to-End Ingestion Integration
#B1.6.2.1 - Successful-path integration
# B1.6.2.1 — Successful execution path orchestration

from pathlib import Path
from uuid import uuid4


def run_ingestion(
    csv_path: Path,
    dataset_root: Path,
    expected_records: int,
    log_path: Path
) -> dict:

    """
    Execute the successful raw ingestion path and
    return the successfully logged provenance record.
    """

    # Step 1: Assign execution identity
    run_id = str(uuid4())

    csv_path = Path(csv_path)
    dataset_root = Path(dataset_root)
    log_path = Path(log_path)

    # Step 2: Extract source metadata
    metadata = extract_source_metadata(
        csv_path,
        dataset_root
    )

    # Step 3: Calculate SHA-256 fingerprint
    sha256_digest = calculate_sha256(csv_path)

    # Step 4: Perform streaming verification
    streaming_result = verify_full_streaming(
        csv_path,
        expected_records
    )

    # Step 5: Build unified provenance record
    provenance_record = build_run_provenance(
        metadata=metadata,
        sha256_digest=sha256_digest,
        streaming_result=streaming_result,
        run_id=run_id
    )

    # Step 6: Append successful ingestion event
    append_ingestion_log(
        provenance_record=provenance_record,
        log_path=log_path
    )

    # Step 7: Return only after successful logging
    return provenance_record

In [15]:
#ensure all five existing component functions have been defined in the current notebook kernel.
required_functions = [
    "extract_source_metadata",
    "calculate_sha256",
    "verify_full_streaming",
    "build_run_provenance",
    "append_ingestion_log"
]

for name in required_functions:
    print(name, ":", callable(globals().get(name)))

extract_source_metadata : True
calculate_sha256 : True
verify_full_streaming : True
build_run_provenance : True
append_ingestion_log : True


In [16]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json

with TemporaryDirectory() as temp_dir:

    # Define directory paths
    dataset_root = Path(temp_dir) / "dataset"
    raw_dir = dataset_root / "raw"

    raw_dir.mkdir(parents=True)

    csv_path = raw_dir / "transactions.csv"

    log_path = Path(temp_dir) / "ingestion_log.jsonl"

    # Create a synthetic CSV fixture
    csv_path.write_text(
        "transaction_id,amount\n"
        "TXN001,100.50\n"
        "TXN002,250.75\n"
        "TXN003,500.00\n",
        encoding="utf-8"
    )

    # Execute the orchestration function
    provenance_record = run_ingestion(
        csv_path=csv_path,
        dataset_root=dataset_root,
        expected_records=3,
        log_path=log_path
    )

    # Display provenance record
    print("PROVENANCE RECORD:\n")
    print(json.dumps(provenance_record, indent=4))

    # Display JSONL log
    print("\nINGESTION LOG:\n")
    print(log_path.read_text(encoding="utf-8"))

PROVENANCE RECORD:

{
    "run_id": "454a4b91-4da5-4e54-8cec-acfb205074d8",
    "record_created_at_utc": "2026-11-25T14:01:35.159812+00:00",
    "source_file": "transactions.csv",
    "relative_path": "raw/transactions.csv",
    "file_size_bytes": 68,
    "last_modified_utc": "2026-11-25T14:01:35.120042+00:00",
    "source_sha256": "5c25ec8c02fb3f211a2cd39996222eb0333bf9200d4b13ac02c3bb71e5426f55",
    "records_processed": 3,
    "total_chunks": 1,
    "reconciliation_status": true,
    "run_status": "success"
}

INGESTION LOG:

{"event_type": "ingestion_success", "logged_at_utc": "2026-11-25T14:01:35.160808+00:00", "run_id": "454a4b91-4da5-4e54-8cec-acfb205074d8", "run_status": "success", "provenance": {"run_id": "454a4b91-4da5-4e54-8cec-acfb205074d8", "record_created_at_utc": "2026-11-25T14:01:35.159812+00:00", "source_file": "transactions.csv", "relative_path": "raw/transactions.csv", "file_size_bytes": 68, "last_modified_utc": "2026-11-25T14:01:35.120042+00:00", "source_sha256": "5

In [17]:
#B1.6.2.2 — Failure-Path Integration
# B1.6.2.2 — Failure-path integration

from pathlib import Path
from uuid import uuid4


def run_ingestion(
    csv_path: Path,
    dataset_root: Path,
    expected_records: int,
    log_path: Path
) -> dict:

    # Generate execution identity
    run_id = str(uuid4())

    csv_path = Path(csv_path)
    dataset_root = Path(dataset_root)
    log_path = Path(log_path)

    try:

        # Stage 1 — Source metadata
        failure_stage = "source_metadata"

        metadata = extract_source_metadata(
            csv_path,
            dataset_root
        )

        # Stage 2 — Source hashing
        failure_stage = "source_hashing"

        sha256_digest = calculate_sha256(csv_path)

        # Stage 3 — Streaming verification
        failure_stage = "streaming_verification"

        streaming_result = verify_full_streaming(
            csv_path,
            expected_records
        )

        # Stage 4 — Provenance construction
        failure_stage = "provenance_construction"

        provenance_record = build_run_provenance(
            metadata=metadata,
            sha256_digest=sha256_digest,
            streaming_result=streaming_result,
            run_id=run_id
        )

    except Exception as error:

        # Append failure event using the same run_id
        append_failed_ingestion_log(
            run_id=run_id,
            failure_stage=failure_stage,
            error=error,
            log_path=log_path
        )

        # Re-raise original processing exception
        raise

    # Stage 5 — Success logging
    # Intentionally outside the try-except block
    append_ingestion_log(
        provenance_record=provenance_record,
        log_path=log_path
    )

    return provenance_record

In [18]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json


with TemporaryDirectory() as temp_dir:

    # Define paths
    dataset_root = Path(temp_dir) / "dataset"
    raw_dir = dataset_root / "raw"

    raw_dir.mkdir(parents=True)

    csv_path = raw_dir / "transactions.csv"

    log_path = Path(temp_dir) / "ingestion_log.jsonl"

    # Create three-record CSV fixture
    csv_path.write_text(
        "transaction_id,amount\n"
        "TXN001,100.50\n"
        "TXN002,250.75\n"
        "TXN003,500.00\n",
        encoding="utf-8"
    )

    # Execute ingestion
    try:

        provenance_record = run_ingestion(
            csv_path=csv_path,
            dataset_root=dataset_root,
            expected_records=5,
            log_path=log_path
        )

        print(json.dumps(provenance_record, indent=4))

    except Exception as error:

        print("Execution failed")
        print("Error type:", type(error).__name__)
        print("Error message:", str(error))

    # Inspect the generated audit log
    if log_path.exists():

        print("\nINGESTION AUDIT LOG:\n")

        for line in log_path.read_text(
            encoding="utf-8"
        ).splitlines():

            print(json.dumps(json.loads(line), indent=4))

Execution failed
Error type: NameError
Error message: name 'append_failed_ingestion_log' is not defined


In [19]:
# B1.6.2.2 — Failure-path integration tests harness

from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import UUID
import json


def test_run_ingestion_failure_path() -> None:

    # Helper function to read JSONL records
    def read_records(log_path):

        return [
            json.loads(line)
            for line in log_path.read_text(
                encoding="utf-8"
            ).splitlines()
        ]

    with TemporaryDirectory() as temp_dir:

        temp_path = Path(temp_dir)

        dataset_root = temp_path / "dataset"

        raw_directory = dataset_root / "raw"

        raw_directory.mkdir(parents=True)

        csv_path = raw_directory / "transactions.csv"

        # ==========================================
        # Create three-record CSV fixture
        # ==========================================

        csv_path.write_text(
            "transaction_id,amount\n"
            "TXN001,100.50\n"
            "TXN002,250.75\n"
            "TXN003,500.00\n",
            encoding="utf-8"
        )

        # Capture original source state
        original_bytes = csv_path.read_bytes()

        original_mtime = csv_path.stat().st_mtime_ns

        # ==========================================
        # T1 — Streaming reconciliation failure
        # Three actual rows, five expected rows
        # ==========================================

        log_path = temp_path / "test_t1.jsonl"

        try:

            run_ingestion(
                csv_path=csv_path,
                dataset_root=dataset_root,
                expected_records=5,
                log_path=log_path
            )

        except ValueError as error:

            original_error_type = type(error).__name__

        else:

            raise AssertionError(
                "T1 FAILED: Expected ValueError"
            )

        # Verify failure event
        records = read_records(log_path)

        assert len(records) == 1

        event = records[0]

        assert event["event_type"] == "ingestion_failure"

        assert event["run_status"] == "failed"

        assert event["failure_stage"] == (
            "streaming_verification"
        )

        assert event["error_type"] == "ValueError"

        assert event["error_type"] == original_error_type

        assert isinstance(event["run_id"], str)

        assert event["run_id"].strip()

        assert UUID(event["run_id"]).version == 4

        # No successful event should exist
        assert not any(
            record["event_type"] == "ingestion_success"
            for record in records
        )

        # Verify source unchanged
        assert csv_path.read_bytes() == original_bytes

        assert csv_path.stat().st_mtime_ns == original_mtime

        print("T1 PASSED: Reconciliation failure logged")

        print(json.dumps(event, indent=4))


        # ==========================================
        # T2 — Missing source CSV
        # ==========================================

        log_path = temp_path / "test_t2.jsonl"

        missing_csv_path = (
            raw_directory / "missing_transactions.csv"
        )

        assert not missing_csv_path.exists()

        try:

            run_ingestion(
                csv_path=missing_csv_path,
                dataset_root=dataset_root,
                expected_records=3,
                log_path=log_path
            )

        except FileNotFoundError as error:

            original_error_type = type(error).__name__

        else:

            raise AssertionError(
                "T2 FAILED: Expected FileNotFoundError"
            )

        # Verify failure event
        records = read_records(log_path)

        assert len(records) == 1

        event = records[0]

        assert event["event_type"] == "ingestion_failure"

        assert event["run_status"] == "failed"

        assert event["failure_stage"] == "source_metadata"

        assert event["error_type"] == "FileNotFoundError"

        assert event["error_type"] == original_error_type

        assert isinstance(event["run_id"], str)

        assert event["run_id"].strip()

        assert UUID(event["run_id"]).version == 4

        # No successful event should exist
        assert not any(
            record["event_type"] == "ingestion_success"
            for record in records
        )

        # Verify missing file was not created
        assert not missing_csv_path.exists()

        # Verify original fixture remains unchanged
        assert csv_path.read_bytes() == original_bytes

        assert csv_path.stat().st_mtime_ns == original_mtime

        print("\nT2 PASSED: Missing source failure logged")

        print(json.dumps(event, indent=4))


        # ==========================================
        # T3 — Successful execution
        # Three actual rows, three expected rows
        # ==========================================

        log_path = temp_path / "test_t3.jsonl"

        provenance = run_ingestion(
            csv_path=csv_path,
            dataset_root=dataset_root,
            expected_records=3,
            log_path=log_path
        )

        # Verify returned provenance
        assert provenance["records_processed"] == 3

        assert provenance["total_chunks"] == 1

        assert provenance["reconciliation_status"] is True

        assert provenance["run_status"] == "success"

        # Verify success event
        records = read_records(log_path)

        assert len(records) == 1

        event = records[0]

        assert event["event_type"] == "ingestion_success"

        assert event["run_status"] == "success"

        # Verify execution identity consistency
        assert (
            event["run_id"]
            == event["provenance"]["run_id"]
            == provenance["run_id"]
        )

        # Verify complete logged provenance
        assert event["provenance"] == provenance

        # No failure event should exist
        assert not any(
            record["event_type"] == "ingestion_failure"
            for record in records
        )

        # Verify original source remains unchanged
        assert csv_path.read_bytes() == original_bytes

        assert csv_path.stat().st_mtime_ns == original_mtime

        print("\nT3 PASSED: Successful execution preserved")

        print(json.dumps(event, indent=4))

    print("\nAll 3 integration tests passed successfully.")

In [22]:
required_functions = [
    "extract_source_metadata",
    "calculate_sha256",
    "verify_full_streaming",
    "build_run_provenance",
    "append_ingestion_log",
    "append_failed_ingestion_log",
    "run_ingestion"
]

for name in required_functions:

    print(
        name,
        ":",
        callable(globals().get(name))
    )

extract_source_metadata : True
calculate_sha256 : True
verify_full_streaming : True
build_run_provenance : True
append_ingestion_log : True
append_failed_ingestion_log : True
run_ingestion : True


In [23]:
test_run_ingestion_failure_path()

T1 PASSED: Reconciliation failure logged
{
    "event_type": "ingestion_failure",
    "logged_at_utc": "2026-11-25T14:36:10.809438+00:00",
    "run_id": "43d0d6b4-7ee0-4647-be1c-017b6a1deebb",
    "run_status": "failed",
    "failure_stage": "streaming_verification",
    "error_type": "ValueError"
}

T2 PASSED: Missing source failure logged
{
    "event_type": "ingestion_failure",
    "logged_at_utc": "2026-11-25T14:36:10.824418+00:00",
    "run_id": "13214fe3-602b-4fa3-89f0-1f85fcbcdc19",
    "run_status": "failed",
    "failure_stage": "source_metadata",
    "error_type": "FileNotFoundError"
}

T3 PASSED: Successful execution preserved
{
    "event_type": "ingestion_success",
    "logged_at_utc": "2026-11-25T14:36:10.839695+00:00",
    "run_id": "01eeaa43-b870-4703-862d-400546b73dc3",
    "run_status": "success",
    "provenance": {
        "run_id": "01eeaa43-b870-4703-862d-400546b73dc3",
        "record_created_at_utc": "2026-11-25T14:36:10.839695+00:00",
        "source_file": "tr